# 📦 LangGraph State

## What is State?
State is the **shared data structure** passed between all nodes in a LangGraph graph.

## State Design Principles
1. State should contain EVERYTHING nodes need
2. Use TypedDict for type safety
3. Use Annotated with operators for merge behavior
4. Keep state serializable (for checkpointing)

## Reducer Functions
When multiple nodes update the same field, you need a **reducer**:
```python
messages: Annotated[list, operator.add]  # appends lists
count: int  # replaces value
```


In [ ]:
# ── State with Reducers ─────────────────────────────────────────────────
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, END
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages

# State with different field behaviors
class ChatState(TypedDict):
    # add_messages reducer: appends new messages, deduplicates by id
    messages: Annotated[list[BaseMessage], add_messages]

    # Simple int: each node REPLACES the value
    token_count: int

    # Annotated with operator.add: ADDS new items to list
    errors: Annotated[list[str], operator.add]

# Nodes return PARTIAL state (only fields they update)
def add_greeting(state: ChatState) -> dict:
    new_msg = AIMessage(content='Hello! How can I help you?')
    return {
        'messages': [new_msg],   # add_messages will append this
        'token_count': 8,
    }

def add_farewell(state: ChatState) -> dict:
    new_msg = AIMessage(content='Goodbye! Have a great day!')
    return {
        'messages': [new_msg],   # add_messages will append again
        'token_count': state['token_count'] + 7,
    }

graph = StateGraph(ChatState)
graph.add_node('greeting', add_greeting)
graph.add_node('farewell', add_farewell)
graph.set_entry_point('greeting')
graph.add_edge('greeting', 'farewell')
graph.add_edge('farewell', END)

app = graph.compile()
result = app.invoke({'messages': [HumanMessage(content='Hi!')], 'token_count': 0, 'errors': []})

print(f'Total messages: {len(result["messages"])}')
for msg in result['messages']:
    print(f'  [{msg.type}]: {msg.content}')
print(f'Token count: {result["token_count"]}')

## 🎯 Interview Questions

1. **What is a reducer in LangGraph state?**
2. **What does add_messages do?**
3. **Why do nodes return partial state instead of full state?**
4. **How do you make LangGraph state serializable?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **LangGraph State — Managing Shared Data**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
